# Q1 · Supervised Learning — Heart Disease Classification
**Notebook:** `q1_supervised.ipynb`  
**Dataset:** `../data/q1_heart_disease.csv`  
**Goal:** Build and evaluate classification models to predict whether a patient has heart disease.
**Target column:** `heart_disease` (1 = disease present, 0 = absent)
**Author:** Gaurav Anand Shukla | BITSoM_BA_25111017

## Task 1 · Data Loading and Inspection — 3 marks

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, precision_score,
                             recall_score, f1_score, accuracy_score)
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Load dataset
df = pd.read_csv('../data/q1_heart_disease.csv')

print('=== Dataset Shape ===')
print(df.shape)

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Value Counts ===')
print(df.isnull().sum())

=== Dataset Shape ===
(800, 12)

=== Data Types ===
age                  int64
sex                  int64
chest_pain_type     object
resting_bp         float64
cholesterol        float64
fasting_bs           int64
resting_ecg         object
max_hr               int64
exercise_angina      int64
oldpeak            float64
st_slope            object
heart_disease        int64
dtype: object

=== Missing Value Counts ===
age                 0
sex                 0
chest_pain_type     0
resting_bp         24
cholesterol        32
fasting_bs          0
resting_ecg         0
max_hr              0
exercise_angina     0
oldpeak             0
st_slope            0
heart_disease       0
dtype: int64


In [2]:
# Display first five rows
df.head()

,age,sex,chest_pain_type,resting_bp,cholesterol,fasting_bs,resting_ecg,max_hr,exercise_angina,oldpeak,st_slope,heart_disease
0,68,0,atypical_angina,142.0,399.0,0,left_ventricular_hypertrophy,169,0,0.4,up,1
1,58,1,non_anginal,163.0,310.0,1,st_t_wave_abnormality,121,1,1.1,up,1
2,44,1,non_anginal,128.0,175.0,0,normal,183,1,0.2,up,0
3,72,1,asymptomatic,114.0,177.0,0,st_t_wave_abnormality,150,0,1.0,up,1
4,37,1,non_anginal,149.0,271.0,0,normal,136,0,0.4,flat,0


### 🔍 Observations
- Dataset has **800 rows and 12 columns** (11 features + 1 target).
- **3 categorical columns**: `chest_pain_type`, `resting_ecg`, `st_slope` — need one-hot encoding.
- **Missing values**: `resting_bp` has **24 nulls** (3%) and `cholesterol` has **32 nulls** (4%) — will impute with median.
- **Target `heart_disease`** is binary (0/1) → this is a **binary classification** problem.
- Class distribution: 407 positive (50.9%) vs 393 negative (49.1%) → nearly balanced, no SMOTE needed.

## Task 2 · Exploratory Data Analysis — 5 marks

In [3]:
# ── Plot 1: Target Class Distribution ──
fig, ax = plt.subplots(figsize=(7, 5))
counts = df['heart_disease'].value_counts()
bars = ax.bar(['No Disease (0)', 'Disease (1)'], counts.values,
              color=['#27AE60', '#E74C3C'], edgecolor='black', width=0.5)
for bar, v in zip(bars, counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
            f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Target Class Distribution — Heart Disease', fontsize=14, fontweight='bold')
ax.set_ylabel('Count')
ax.set_ylim(0, 470)
plt.tight_layout()
plt.savefig('target_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Target distribution: No Disease =', counts[0], '| Disease =', counts[1])

Target distribution: No Disease = 393 | Disease = 407


### 📊 Plot 1 Interpretation — Target Class Distribution
The dataset is **nearly perfectly balanced**: 393 patients without heart disease (49.1%) and 407 with heart disease (50.9%). This is an ideal distribution for classification — we do **not** need class-balancing techniques (SMOTE, class weights). **Accuracy is a valid metric** alongside precision/recall because neither class dominates.

In [4]:
# ── Plot 2: Correlation Heatmap ──
num_cols = ['age','sex','resting_bp','cholesterol','fasting_bs',
            'max_hr','exercise_angina','oldpeak','heart_disease']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.show()
print('Correlation with heart_disease:')
print(corr['heart_disease'].drop('heart_disease').sort_values(ascending=False).round(3))

Correlation with heart_disease:
exercise_angina    0.474
oldpeak            0.406
age                0.272
sex                0.244
fasting_bs         0.116
resting_bp         0.072
cholesterol       -0.055
max_hr            -0.373
Name: heart_disease, dtype: float64


### 📊 Plot 2 Interpretation — Correlation Heatmap
- **`exercise_angina`** (+0.474) and **`oldpeak`** (+0.406) are the strongest positive predictors — exercise-induced angina and ST-depression strongly indicate heart disease.
- **`max_hr`** (–0.373) is the strongest negative predictor — higher peak heart rate indicates better cardiovascular fitness and lower disease risk.
- **`age`** (+0.272) and **`sex`** (+0.244) show moderate positive correlation — older patients and males face higher risk.
- **`cholesterol`** shows a surprisingly weak negative correlation (–0.055) — this suggests its clinical value here is limited without interaction terms.

In [5]:
# ── Plot 3: Age Distribution by Heart Disease Status ──
fig, ax = plt.subplots(figsize=(10, 5))
for status, color, label in [(0,'#27AE60','No Disease'),(1,'#E74C3C','Disease')]:
    data = df[df['heart_disease']==status]['age']
    ax.hist(data, bins=20, alpha=0.6, color=color, label=label, edgecolor='black')
    ax.axvline(data.mean(), color=color, linestyle='--', linewidth=2,
               label=f'Mean ({label}): {data.mean():.1f}')
ax.set_title('Age Distribution by Heart Disease Status', fontsize=14, fontweight='bold')
ax.set_xlabel('Age'); ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('age_distribution.png', dpi=100, bbox_inches='tight')
plt.show()
print('Mean age — No Disease:', round(df[df.heart_disease==0]['age'].mean(),1))
print('Mean age — Disease   :', round(df[df.heart_disease==1]['age'].mean(),1))

Mean age — No Disease: 50.6
Mean age — Disease   : 53.6


### 📊 Plot 3 Interpretation — Age Distribution
Patients **with heart disease** are on average **3 years older** (mean 53.6 vs 50.6). Both distributions peak between 50–65 years, reflecting the clinical population studied. Age alone is not a definitive predictor but contributes meaningfully when combined with other risk factors — this validates its inclusion as a feature.

In [6]:
# ── Plot 4: Oldpeak Distribution by Target (Box Plot) ──
fig, ax = plt.subplots(figsize=(8, 5))
df.boxplot(column='oldpeak', by='heart_disease', ax=ax,
           boxprops=dict(color='#2980B9'),
           medianprops=dict(color='#E74C3C', linewidth=2))
ax.set_title('Oldpeak (ST Depression) by Heart Disease Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Heart Disease (0=No, 1=Yes)')
ax.set_ylabel('Oldpeak value')
plt.suptitle('')
plt.tight_layout()
plt.savefig('oldpeak_boxplot.png', dpi=100, bbox_inches='tight')
plt.show()
print('Oldpeak median — No Disease:', round(df[df.heart_disease==0]['oldpeak'].median(),2))
print('Oldpeak median — Disease   :', round(df[df.heart_disease==1]['oldpeak'].median(),2))

Oldpeak median — No Disease: 0.3
Oldpeak median — Disease   : 1.2


### 📊 Plot 4 Interpretation — Oldpeak Box Plot
The box plot reveals a **clear separation** in ST-depression (oldpeak) between the two classes. Disease patients have a median oldpeak of **1.2** vs **0.3** for healthy patients — a 4× difference. The interquartile ranges barely overlap, making `oldpeak` one of the most discriminating features. This aligns with medical knowledge: ST segment depression during exercise is a key diagnostic indicator of ischaemic heart disease.

## Task 3 · Data Preprocessing — 5 marks

In [7]:
# Step 1 — Handle Missing Values with Median Imputation
print('Missing values BEFORE imputation:')
print(df[['resting_bp','cholesterol']].isnull().sum())

df['resting_bp']  = df['resting_bp'].fillna(df['resting_bp'].median())
df['cholesterol'] = df['cholesterol'].fillna(df['cholesterol'].median())

print('\nMissing values AFTER imputation:')
print(df[['resting_bp','cholesterol']].isnull().sum())
print('\nresting_bp median used:', df['resting_bp'].median())
print('cholesterol median used:', df['cholesterol'].median())

Missing values BEFORE imputation:
resting_bp     24
cholesterol    32
dtype: int64

Missing values AFTER imputation:
resting_bp     0
cholesterol    0
dtype: int64

resting_bp median used: 130.0
cholesterol median used: 247.0


### ✅ Justification — Median Imputation
**Median** was chosen over mean or row-dropping for three reasons:
1. **Robustness to outliers**: Clinical measurements like `resting_bp` and `cholesterol` often contain extreme values. The median is unaffected by outliers — unlike the mean which gets pulled toward them.
2. **Data preservation**: Only 3–4% of rows are affected. Dropping them would waste 56 valuable records unnecessarily.
3. **Clinical validity**: The median of 130 mmHg (resting_bp) and 247 mg/dL (cholesterol) represent the central tendency of the patient population and are physiologically sensible imputed values.

In [8]:
# Step 2 — One-Hot Encoding of Categorical Variables
cat_cols = ['chest_pain_type', 'resting_ecg', 'st_slope']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=False)

print('Shape after one-hot encoding:', df_encoded.shape)
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print('New dummy columns:', new_cols)

Shape after one-hot encoding: (800, 19)
New dummy columns: ['chest_pain_type_asymptomatic', 'chest_pain_type_atypical_angina', 'chest_pain_type_non_anginal', 'chest_pain_type_typical_angina', 'resting_ecg_left_ventricular_hypertrophy', 'resting_ecg_normal', 'resting_ecg_st_t_wave_abnormality', 'st_slope_down', 'st_slope_flat', 'st_slope_up']


In [9]:
# Step 3 — Feature / Target Split + Stratified Train-Test Split
X = df_encoded.drop('heart_disease', axis=1)
y = df_encoded['heart_disease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print('Training set size:', X_train.shape)
print('Test set size    :', X_test.shape)
print('\nClass distribution — Training set:')
print(y_train.value_counts())
print('\nClass distribution — Test set:')
print(y_test.value_counts())

Training set size: (640, 18)
Test set size    : (160, 18)

Class distribution — Training set:
heart_disease
1    326
0    314
dtype: int64

Class distribution — Test set:
heart_disease
1    81
0    79
dtype: int64


In [10]:
# Step 4 — Feature Scaling with StandardScaler
# Fit ONLY on training data to prevent data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # apply training statistics to test

print('Scaling complete — no data leakage.')
print('Mean of first 3 train features (should be ~0):', X_train_scaled.mean(axis=0)[:3].round(4))
print('Std  of first 3 train features (should be ~1):', X_train_scaled.std(axis=0)[:3].round(4))

Scaling complete — no data leakage.
Mean of first 3 train features (should be ~0): [-0.  0.  0.]
Std  of first 3 train features (should be ~1): [1. 1. 1.]


## Task 4 · Model Training — 5 marks

In [11]:
# Train all three models with random_state=42
dt = DecisionTreeClassifier(random_state=42)
rf = RandomForestClassifier(random_state=42)
gb = GradientBoostingClassifier(random_state=42)

dt.fit(X_train_scaled, y_train)
rf.fit(X_train_scaled, y_train)
gb.fit(X_train_scaled, y_train)

print('✓ Decision Tree Classifier    — trained')
print('✓ Random Forest Classifier    — trained')
print('✓ Gradient Boosting Classifier — trained')

✓ Decision Tree Classifier    — trained
✓ Random Forest Classifier    — trained
✓ Gradient Boosting Classifier — trained


## Task 5 · Model Evaluation — 6 marks

In [12]:
models = {'Decision Tree': dt, 'Random Forest': rf, 'Gradient Boosting': gb}
results = {}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, model) in zip(axes, models.items()):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)
    results[name] = {'Accuracy':acc,'Precision':prec,'Recall':rec,'F1-Score':f1}

    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=['No Disease','Disease'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}',
                 fontsize=10, fontweight='bold')

plt.suptitle('Confusion Matrices — All Three Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()
print('Confusion matrices saved.')

Confusion matrices saved.


In [13]:
# Detailed classification reports
for name, model in models.items():
    y_pred = model.predict(X_test_scaled)
    print(f"\n{'='*55}")
    print(f"Classification Report — {name}")
    print('='*55)
    print(classification_report(y_test, y_pred,
                                target_names=['No Disease','Disease']))


Classification Report — Decision Tree
              precision    recall  f1-score   support

  No Disease       0.71      0.71      0.71        79
     Disease       0.73      0.73      0.73        81

    accuracy                           0.72       160
   macro avg       0.72      0.72      0.72       160
weighted avg       0.72      0.72      0.72       160


Classification Report — Random Forest
              precision    recall  f1-score   support

  No Disease       0.79      0.76      0.77        79
     Disease       0.77      0.81      0.79        81

    accuracy                           0.79       160
   macro avg       0.78      0.79      0.78       160
weighted avg       0.78      0.79      0.78       160


Classification Report — Gradient Boosting
              precision    recall  f1-score   support

  No Disease       0.77      0.77      0.77        79
     Disease       0.78      0.78      0.78        81

    accuracy                           0.78       160
   macr

In [14]:
# Summary performance table
results_df = pd.DataFrame(results).T.round(4)
print('\nModel Performance Summary:')
print(results_df.to_string())


Model Performance Summary:
                   Accuracy  Precision  Recall  F1-Score
Decision Tree        0.7188     0.7195  0.7284    0.7239
Random Forest        0.7875     0.7765  0.8148    0.7952
Gradient Boosting    0.7750     0.7778  0.7778    0.7778


### 🏆 Best Model Analysis
**Random Forest Classifier** is the best-performing model:
- **Highest Accuracy**: 78.75%
- **Highest F1-Score**: 0.7952 — best harmonic mean of precision and recall
- **Highest Recall**: 81.48% — critical in medical diagnosis (missing a true positive is clinically dangerous)

**Why Random Forest outperforms the others:**
1. **Ensemble averaging**: Aggregates 100 decision trees, drastically reducing variance compared to a single Decision Tree (72%).
2. **Feature randomness**: Random feature subsets at each split prevent correlated trees and overfitting.
3. **Clinical priority**: In heart disease detection, **Recall (sensitivity)** matters most — a false negative (predicting 'no disease' for a sick patient) is far more dangerous than a false alarm. RF achieves the highest recall at 81.48%.

**Conclusion**: Random Forest is selected for hyperparameter tuning.

## Task 6 · Hyperparameter Tuning — GridSearchCV on Random Forest — 4 marks

In [15]:
param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [None, 10],
    'min_samples_split': [2, 5],
    'max_features'     : ['sqrt']
}

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator  = RandomForestClassifier(random_state=42),
    param_grid = param_grid,
    cv         = cv_strategy,
    scoring    = 'f1',
    n_jobs     = -1,
    verbose    = 0
)
grid_search.fit(X_train_scaled, y_train)

print('Best Parameters Found by GridSearchCV:')
print(grid_search.best_params_)
print(f'\nBest Cross-Validated F1-Score: {grid_search.best_score_:.4f}')

Best Parameters Found by GridSearchCV:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200}

Best Cross-Validated F1-Score: 0.8255


In [16]:
# Compare tuned vs baseline Random Forest
best_rf    = grid_search.best_estimator_
y_pred_base  = rf.predict(X_test_scaled)
y_pred_tuned = best_rf.predict(X_test_scaled)

print(f"{'Metric':<20} {'Baseline RF':>14} {'Tuned RF':>12}")
print('-'*48)
metrics = [('Accuracy', accuracy_score), ('Precision', precision_score),
           ('Recall', recall_score), ('F1-Score', f1_score)]
for mname, fn in metrics:
    b = fn(y_test, y_pred_base)
    t = fn(y_test, y_pred_tuned)
    print(f"{mname:<20} {b:>14.4f} {t:>12.4f}")

print('\nTuned Model Confusion Matrix:')
print(confusion_matrix(y_test, y_pred_tuned))

Metric                  Baseline RF     Tuned RF
------------------------------------------------
Accuracy                     0.7875       0.7875
Precision                    0.7765       0.7701
Recall                       0.8148       0.8272
F1-Score                     0.7952       0.7976

Tuned Model Confusion Matrix:
[[59 20]
 [14 67]]


### 📈 Hyperparameter Tuning Results
GridSearchCV with 5-fold stratified cross-validation identified: **max_depth=10, max_features='sqrt', min_samples_split=5, n_estimators=200**.

**Key improvement**: Recall improved from **81.48% → 82.72%** — meaning the tuned model catches more true heart disease cases, which is the most clinically important metric.

**Why the gain is modest**: The dataset has 800 samples; with strong baseline parameters (sklearn defaults are well-tuned), gains from grid search are incremental. The real value of GridSearchCV is that it uses **5-fold cross-validation** to select parameters that generalise, not overfit. The best CV F1 of 0.8255 is close to the test F1 of 0.7976, confirming **no overfitting** in the final model.

**Tuned confusion matrix**: 59 TN, 20 FP, 14 FN, 67 TP — the tuned model correctly identifies 67 out of 81 disease cases.